COM7022 Machine Learning - Product clustering for an e-commerce fragrance catalogue
Dataset: Sales-[5444].csv

Pipeline
  1. Data audit
  2. Cleaning and recovery of missing values from the free-text fields
  3. Feature engineering and selection
  4. Transformation (log, scaling) and comparison of scalers / feature sets
  5. Clustering: K-Means, Agglomerative (Ward), DBSCAN, Gaussian Mixture
  6. Validation: silhouette, Davies-Bouldin, Calinski-Harabasz, bootstrap stability, cross-model ARI
  7. PCA for dimensionality reduction and visualisation
  8. Cluster profiling for the business


In [2]:
import io, json, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)

warnings.filterwarnings("ignore")
DATA, OUT, SEED = "Sales-[5444].csv", "figs/", 42
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10.5, "axes.titlesize": 12.5,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e8e8e8", "axes.axisbelow": True,
                     "savefig.dpi": 200, "savefig.bbox": "tight"})
INK, MUTED = "#1f2933", "#7b8794"
CL_COL = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9", "#F0E442"]
def save(fig, name): fig.savefig(OUT + name, facecolor="white"); plt.close(fig)

# =============================================================== 1. AUDIT
raw = pd.read_csv(DATA)
log = io.StringIO()
log.write(f">>> raw.shape\n{raw.shape}\n\n>>> raw.isna().sum()\n{raw.isna().sum().to_string()}\n\n")
log.write(f">>> raw.duplicated().sum()\n{raw.duplicated().sum()}\n\n")
log.write(f">>> raw['Catatory'].nunique()   # spelling variants of product type\n{raw['Catatory'].nunique()}\n\n")
log.write(">>> raw['text'] patterns\n" + raw["text"].str.replace(r"\d[\d,]*", "N", regex=True)
          .value_counts().head(5).to_string() + "\n\n")

# =============================================================== 2. CLEANING
df = raw.copy()
num = lambda s: pd.to_numeric(s.str.replace(",", ""), errors="coerce")

# 2a. price: recover from priceWithCurrency; convert the three CAD listings to USD
pwc = num(df["priceWithCurrency"].str.extract(r"\$([\d,.]+)")[0])
n_price_rec = int((df["price"].isna() & pwc.notna()).sum())
df["price"] = df["price"].fillna(pwc)
CAD_USD = 0.73   # approximate May-2024 exchange rate
cad = df["priceWithCurrency"].str.startswith("C", na=False)
df.loc[cad, "price"] = df.loc[cad, "price"] * CAD_USD

# 2b. sold: recover from the 'text' field ("... / 116 sold")
sold_txt = num(df["text"].str.extract(r"/\s*([\d,]+)\s*sold")[0])
n_sold_rec = int((df["sold"].isna() & sold_txt.notna()).sum())
df["sold"] = df["sold"].fillna(sold_txt)

# 2c. availability: the field is right-censored ("More than 10 available" is stored as 10),
#     and some listings only say "Last One" or "Limited quantity". Convert to an ordinal stock band.
t = df["text"].fillna("")
df["stock_band"] = np.select(
    [t.str.contains("Last One"), t.str.contains("Limited quantity"),
     t.str.contains("More than"), df["available"] > 10, df["available"] <= 5, df["available"] <= 10],
    [0, 1, 3, 3, 1, 2], default=np.nan)
STOCK_LBL = {0: "Last one", 1: "Low (2-5 / limited)", 2: "Medium (6-10)", 3: "High (>10)"}

# 2d. product type: 63 raw labels -> 5 concentration groups, using 'details' when the label is vague
def concentration(cat, det):
    s = f"{cat} {det}".lower() if str(cat).lower() in {"nan", "/", "y", "does not apply", "fragrances", "various", "assorted", "fragrance & perfume"} else str(cat).lower()
    if re.search(r"extrait|elixir|\bparfum\b(?! de)|le parfum|parfum intense|pure perfume", s) and "eau de parfum" not in s: return "Parfum / Extrait"
    if re.search(r"eau de parfum|eau de perfume|\bedp\b|perfume", s): return "Eau de Parfum"
    if re.search(r"eau de toil|\bedt\b|toillette", s): return "Eau de Toilette"
    if re.search(r"cologne|\bedc\b|aftershave|splash", s): return "Cologne / Aftershave"
    return "Other (oils, sprays, sets)"
df["concentration"] = [concentration(c, d) for c, d in zip(df["Catatory"], df["details"])]

# 2e. auxiliary attributes used for profiling
d = df["details"].str.lower()
oz = num(d.str.extract(r"(\d+(?:\.\d+)?)\s*(?:fl\.?\s*)?oz")[0])
ml = num(d.str.extract(r"(\d+(?:\.\d+)?)\s*ml")[0])
df["volume_ml"] = ml.fillna(oz * 29.5735)
df["international"] = (~df["itemLocation"].str.strip().str.endswith("United States")).astype(int)
df["brand"] = df["Item_Name"].fillna("Unknown").str.strip()

# 2f. drop records that cannot be clustered (no sales or no stock information)
before = len(df)
df = df.dropna(subset=["price", "sold", "stock_band"]).reset_index(drop=True)
dropped = before - len(df)
log.write(f">>> recovered price from priceWithCurrency: {n_price_rec}\n")
log.write(f">>> recovered sold from text: {n_sold_rec}\n")
log.write(f">>> CAD prices converted to USD: {int(cad.sum())}\n")
log.write(f">>> records dropped (no sales/stock data): {dropped}\n>>> clean.shape {df.shape}\n")
open(OUT + "audit.txt", "w").write(log.getvalue())
fig = plt.figure(figsize=(7.8, 8.6)); fig.patch.set_facecolor("#1e1e1e")
fig.text(0.02, 0.98, log.getvalue(), family="DejaVu Sans Mono", fontsize=8.3, color="#d4d4d4", va="top")
fig.savefig(OUT + "fig01_audit.png", facecolor="#1e1e1e"); plt.close(fig)

# =============================================================== 3. FEATURES
df["log_price"] = np.log1p(df["price"])
df["log_sold"] = np.log1p(df["sold"])
df["revenue"] = df["price"] * df["sold"]          # gross merchandise value (GMV) proxy
CORE = ["log_price", "log_sold", "stock_band"]     # final clustering features

# EDA figure: raw vs log distributions
fig, ax = plt.subplots(2, 2, figsize=(10, 6.4))
for j, (c, lab) in enumerate([("price", "Price (USD)"), ("sold", "Units sold")]):
    ax[0, j].hist(df[c], bins=40, color="#9fb3c8", edgecolor="white")
    ax[0, j].set_title(f"{lab}: raw (skew = {df[c].skew():.1f})", loc="left", fontsize=11)
    ax[1, j].hist(np.log1p(df[c]), bins=40, color="#0072B2", edgecolor="white")
    ax[1, j].set_title(f"{lab}: log(1+x) (skew = {np.log1p(df[c]).skew():.1f})", loc="left", fontsize=11)
    ax[1, j].set_xlabel(f"log(1 + {c})")
fig.suptitle("Distributions before and after log transformation", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "fig02_dist.png")

# Category and stock figure
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"wspace": 0.6})
cc = df["concentration"].value_counts().sort_values()
a1.barh(cc.index, cc.values, color="#0072B2")
for i, v in enumerate(cc.values): a1.text(v + 5, i, str(v), va="center", fontsize=9)
a1.set_title(f"(a) 63 raw labels mapped to 5 groups", loc="left", fontsize=11); a1.grid(axis="y", visible=False)
sb = df["stock_band"].map(STOCK_LBL).value_counts().reindex(STOCK_LBL.values())
a2.barh(sb.index, sb.values, color="#E69F00")
for i, v in enumerate(sb.values): a2.text(v + 5, i, str(v), va="center", fontsize=9)
a2.set_title("(b) Ordinal stock band", loc="left", fontsize=11); a2.grid(axis="y", visible=False)
save(fig, "fig03_cats.png")

# Correlation of candidate features
cand = pd.DataFrame({"log price": df["log_price"], "log sold": df["log_sold"], "stock band": df["stock_band"],
                     "log volume (ml)": np.log1p(df["volume_ml"]), "international": df["international"],
                     "EDP / Parfum": df["concentration"].isin(["Eau de Parfum", "Parfum / Extrait"]).astype(int)})
fig, ax = plt.subplots(figsize=(6.8, 5.4))
sns.heatmap(cand.corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            linewidths=.6, cbar_kws={"label": "Spearman rho", "shrink": .8}, ax=ax)
ax.set_title("Spearman correlation of candidate features", loc="left"); ax.grid(False)
plt.xticks(rotation=35, ha="right"); save(fig, "fig04_corr.png")

# =============================================================== 4. TRANSFORMATION EXPERIMENTS
def km_scores(X, k):
    lab = KMeans(k, n_init=20, random_state=SEED).fit_predict(X)
    return silhouette_score(X, lab), davies_bouldin_score(X, lab), lab
def diagnose(lab):
    """What is the partition really driven by? smallest cluster + agreement with single variables."""
    return (int(np.bincount(lab).min()), round(adjusted_rand_score(df["stock_band"], lab), 2),
            round(adjusted_rand_score(df["concentration"], lab), 2))
onehot = pd.get_dummies(df["concentration"], prefix="", dtype=float)
FS = {"A: core (price, sold, stock)": df[CORE].values,
      "B: core + concentration one-hot + origin": np.column_stack([df[CORE].values, onehot.values, df["international"].values])}
SC = {"StandardScaler (z-score)": StandardScaler(), "RobustScaler (median/IQR)": RobustScaler(),
      "MinMaxScaler (normalise 0-1)": MinMaxScaler()}
rows = []
for fsn, Xr in FS.items():
    for scn, sc in SC.items():
        if fsn.startswith("B") and not scn.startswith("Standard"): continue
        Xs = sc.fit_transform(Xr)
        best = max(((k, *km_scores(Xs, k)) for k in range(2, 9)), key=lambda r: r[1])
        rows.append([fsn, scn, Xs.shape[1], best[0], round(best[1], 3), round(best[2], 3), *diagnose(best[3])])
# also without log transform
Xs = StandardScaler().fit_transform(df[["price", "sold", "stock_band"]])
best = max(((k, *km_scores(Xs, k)) for k in range(2, 9)), key=lambda r: r[1])
rows.append(["A without log transform", "StandardScaler (z-score)", 3, best[0], round(best[1], 3), round(best[2], 3), *diagnose(best[3])])
trans = pd.DataFrame(rows, columns=["Feature set", "Scaling", "Dims", "Best k", "Silhouette", "Davies-Bouldin",
                                    "Smallest cluster", "ARI vs stock band", "ARI vs concentration"])
trans.to_csv(OUT + "tab_transform.csv", index=False)

# Final design matrix
scaler = StandardScaler()
X = scaler.fit_transform(df[CORE])

# =============================================================== 5. K-MEANS
ks = range(2, 11); inert, sil, dbi, chi = [], [], [], []
for k in ks:
    m = KMeans(k, n_init=20, random_state=SEED).fit(X)
    inert.append(m.inertia_); sil.append(silhouette_score(X, m.labels_))
    dbi.append(davies_bouldin_score(X, m.labels_)); chi.append(calinski_harabasz_score(X, m.labels_))
kdf = pd.DataFrame({"k": list(ks), "inertia": inert, "silhouette": sil, "davies_bouldin": dbi, "calinski": chi}).round(3)
kdf.to_csv(OUT + "tab_kmeans_k.csv", index=False)
K = 5   # chosen: better silhouette/DB than k=4, isolates the stock-out-risk bestsellers; k>=6 adds only price splits
fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.8))
ax[0].plot(ks, inert, "o-", color="#0072B2"); ax[0].set_title("Elbow (inertia)", loc="left"); ax[0].set_xlabel("k")
ax[1].plot(ks, sil, "o-", color="#009E73"); ax[1].set_title("Silhouette (higher = better)", loc="left"); ax[1].set_xlabel("k")
ax[2].plot(ks, dbi, "o-", color="#D55E00"); ax[2].set_title("Davies-Bouldin (lower = better)", loc="left"); ax[2].set_xlabel("k")
for a in ax: a.axvline(K, color=MUTED, ls=":", lw=1.3)
ax[0].annotate(f"chosen k = {K}", (K, inert[K - 2]), xytext=(6, inert[1]), arrowprops=dict(arrowstyle="->", color=MUTED), fontsize=9)
fig.tight_layout(); save(fig, "fig05_kmeans_k.png")
km = KMeans(K, n_init=50, random_state=SEED).fit(X)

# =============================================================== 6. HIERARCHICAL
link_rows = []
for lk in ["ward", "complete", "average"]:
    lab = AgglomerativeClustering(K, linkage=lk).fit_predict(X)
    sizes = np.bincount(lab)
    link_rows.append([lk, round(silhouette_score(X, lab), 3), round(davies_bouldin_score(X, lab), 3), int(sizes.min()), int(sizes.max())])
pd.DataFrame(link_rows, columns=["Linkage", "Silhouette", "Davies-Bouldin", "Smallest cluster", "Largest cluster"]).to_csv(OUT + "tab_linkage.csv", index=False)
hc = AgglomerativeClustering(K, linkage="ward").fit(X)
Z = linkage(X, method="ward")
fig, ax = plt.subplots(figsize=(10.5, 4.3))
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=Z[-(K - 1), 2], above_threshold_color=MUTED,
           leaf_font_size=8, ax=ax)
ax.axhline(Z[-(K - 1), 2] - 0.5, color="#D55E00", ls="--", lw=1.2)
ax.text(1, Z[-(K - 1), 2] + 1, f"cut for {K} clusters", color="#D55E00", fontsize=9)
ax.set_title("Ward-linkage dendrogram (truncated to the last 30 merges)", loc="left")
ax.set_ylabel("Ward distance"); ax.set_xlabel("Cluster size (in brackets) or product index"); ax.grid(False)
save(fig, "fig06_dendrogram.png")

# =============================================================== 7. DBSCAN
MINPTS = 2 * X.shape[1]   # rule of thumb: 2 x dimensions
kd = np.sort(NearestNeighbors(n_neighbors=MINPTS).fit(X).kneighbors(X)[0][:, -1])
grid = []
for eps in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    for mp in [4, 6, 10, 15]:
        lab = DBSCAN(eps=eps, min_samples=mp).fit_predict(X)
        ncl = len(set(lab)) - (-1 in lab); noise = (lab == -1).mean()
        s = silhouette_score(X[lab != -1], lab[lab != -1]) if ncl > 1 else np.nan
        grid.append([eps, mp, ncl, round(noise * 100, 1), round(s, 3) if s == s else np.nan])
gdf = pd.DataFrame(grid, columns=["eps", "min_samples", "clusters", "noise_%", "silhouette_non_noise"])
gdf.to_csv(OUT + "tab_dbscan_grid.csv", index=False)
EPS = 0.5
db = DBSCAN(eps=EPS, min_samples=MINPTS).fit(X)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(kd, color="#0072B2"); ax.axhline(EPS, color="#D55E00", ls="--")
ax.text(20, EPS + 0.05, f"eps = {EPS}", color="#D55E00", fontsize=9)
ax.set_title(f"DBSCAN k-distance plot (k = min_samples = {MINPTS})", loc="left")
ax.set_xlabel("Products sorted by distance"); ax.set_ylabel(f"Distance to {MINPTS}th neighbour")
save(fig, "fig07_kdist.png")

# =============================================================== 8. GAUSSIAN MIXTURE
bic = [GaussianMixture(k, covariance_type="full", random_state=SEED, n_init=5).fit(X).bic(X) for k in ks]
gmm = GaussianMixture(K, covariance_type="full", random_state=SEED, n_init=5).fit(X)
gm_lab = gmm.predict(X)
pd.DataFrame({"k": list(ks), "BIC": np.round(bic, 1)}).to_csv(OUT + "tab_gmm_bic.csv", index=False)

# =============================================================== 9. VALIDATION & COMPARISON
def stability(make, B=30):
    """Bootstrap stability: refit on 80% subsamples and compare labels with the full fit (ARI)."""
    full = make().fit_predict(X); rng = np.random.default_rng(SEED); out = []
    for _ in range(B):
        idx = rng.choice(len(X), int(.8 * len(X)), replace=False)
        out.append(adjusted_rand_score(full[idx], make().fit_predict(X[idx])))
    return np.mean(out), np.std(out)
models = {
    f"K-Means (k={K})": (km.labels_, lambda: KMeans(K, n_init=20, random_state=SEED)),
    f"Agglomerative Ward (k={K})": (hc.labels_, lambda: AgglomerativeClustering(K, linkage="ward")),
    f"DBSCAN (eps={EPS}, minPts={MINPTS})": (db.labels_, lambda: DBSCAN(eps=EPS, min_samples=MINPTS)),
    f"Gaussian Mixture (k={K})": (gm_lab, lambda: GaussianMixture(K, random_state=SEED, n_init=3)),
}
comp = []
for name, (lab, make) in models.items():
    mask = lab != -1; ncl = len(set(lab[mask]))
    st_m, st_s = stability(make)
    sizes = np.bincount(lab[mask])
    comp.append([name, ncl, round((~mask).mean() * 100, 1), round(silhouette_score(X[mask], lab[mask]), 3),
                 round(davies_bouldin_score(X[mask], lab[mask]), 3), round(calinski_harabasz_score(X[mask], lab[mask]), 1),
                 f"{st_m:.2f} ± {st_s:.2f}", f"{sizes.min()}-{sizes.max()}",
                 round(adjusted_rand_score(df["stock_band"][mask], lab[mask]), 2)])
comp = pd.DataFrame(comp, columns=["Model", "Clusters", "Noise %", "Silhouette", "Davies-Bouldin", "Calinski-Harabasz", "Stability (ARI)", "Cluster sizes", "ARI vs stock band"])
comp.to_csv(OUT + "tab_compare.csv", index=False)
names = list(models)
ari = pd.DataFrame([[adjusted_rand_score(models[a][0], models[b][0]) for b in names] for a in names],
                   index=[n.split(" (")[0] for n in names], columns=[n.split(" (")[0] for n in names]).round(2)
ari.to_csv(OUT + "tab_ari.csv")

# =============================================================== 10. PCA
pca = PCA().fit(X); P = pca.transform(X)
evr = pca.explained_variance_ratio_
load = pd.DataFrame(pca.components_[:2].T, index=["log price", "log sold", "stock band"], columns=["PC1", "PC2"]).round(3)
load.to_csv(OUT + "tab_loadings.csv")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
a1.bar(["PC1", "PC2", "PC3"], evr * 100, color="#0072B2")
a1.plot(["PC1", "PC2", "PC3"], np.cumsum(evr) * 100, "o-", color="#D55E00", label="cumulative")
for i, v in enumerate(evr): a1.text(i, v * 100 + 2, f"{v*100:.1f}%", ha="center", fontsize=9)
a1.set_ylim(0, 108); a1.set_ylabel("Variance explained (%)"); a1.legend(frameon=False, loc="center right")
a1.set_title("(a) Scree plot", loc="left", fontsize=11)
a2.scatter(P[:, 0], P[:, 1], s=6, color="#c5ced8")
for f, (x, y) in load.iterrows():
    a2.arrow(0, 0, x * 3, y * 3, color="#D55E00", width=.02, head_width=.12)
    a2.text(x * 3.4, y * 3.4, f, color="#D55E00", fontsize=10, ha="center", fontweight="bold")
a2.set_xlabel(f"PC1 ({evr[0]*100:.0f}%)"); a2.set_ylabel(f"PC2 ({evr[1]*100:.0f}%)")
a2.set_title("(b) Biplot: feature loadings on PC1-PC2", loc="left", fontsize=11)
fig.tight_layout(); save(fig, "fig08_pca.png")

# =============================================================== 11. CLUSTER NAMING (K-Means) & PCA VIEWS
df["cluster"] = km.labels_
prof = df.groupby("cluster").agg(Products=("price", "size"), Median_price=("price", "median"),
                                 Median_sold=("sold", "median"), Mean_stock_band=("stock_band", "mean"),
                                 Revenue=("revenue", "sum"))
# name clusters from their centroid position (deterministic, independent of label numbering)
c = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_), columns=CORE)
NAMES = {}
prem = c.log_price.idxmax(); NAMES[prem] = "Premium niche"
rest = c.drop(prem)
best2 = rest.log_sold.nlargest(2).index
NAMES[rest.loc[best2, "stock_band"].idxmax()] = "Well-stocked bestsellers"
NAMES[rest.loc[best2, "stock_band"].idxmin()] = "Bestsellers at stock-out risk"
slow = rest.drop(best2)
NAMES[slow.stock_band.idxmax()] = "Overstocked slow sellers"
NAMES[slow.stock_band.idxmin()] = "Low-stock slow sellers"
df["segment"] = df["cluster"].map(NAMES)
SEG_ORDER = ["Bestsellers at stock-out risk", "Well-stocked bestsellers", "Premium niche",
             "Overstocked slow sellers", "Low-stock slow sellers"]
SEG_COL = dict(zip(SEG_ORDER, CL_COL))

fig, ax = plt.subplots(2, 2, figsize=(11.5, 9.2), sharex=True, sharey=True)
for a, (nm, (lab, _)) in zip(ax.ravel(), models.items()):
    if nm.startswith("K-Means"):
        for s in SEG_ORDER:
            m = df["segment"] == s; a.scatter(P[m, 0], P[m, 1], s=9, alpha=.7, color=SEG_COL[s], label=s)
        a.legend(frameon=False, fontsize=8, markerscale=1.8, loc="lower left")
    else:
        for j, l in enumerate(sorted(set(lab))):
            m = lab == l
            a.scatter(P[m, 0], P[m, 1], s=9 if l != -1 else 14, alpha=.7, marker="o" if l != -1 else "x",
                      color=CL_COL[j % 7] if l != -1 else "#222", label="noise" if l == -1 else None)
        if -1 in lab: a.legend(frameon=False, fontsize=8, loc="lower left")
    r = comp.set_index("Model").loc[nm]
    a.set_title(f"{nm}\nsilhouette {r.Silhouette:.3f} | clusters {r.Clusters} | noise {r['Noise %']}%", loc="left", fontsize=10.5)
for a in ax[1]: a.set_xlabel(f"PC1 ({evr[0]*100:.0f}%)")
for a in ax[:, 0]: a.set_ylabel(f"PC2 ({evr[1]*100:.0f}%)")
fig.tight_layout(); save(fig, "fig09_pca_models.png")

# silhouette plot for K-Means
sv = silhouette_samples(X, km.labels_)
fig, ax = plt.subplots(figsize=(8.5, 4.6)); y0 = 10
for s in SEG_ORDER:
    v = np.sort(sv[df["segment"] == s]); ax.fill_betweenx(np.arange(y0, y0 + len(v)), 0, v, color=SEG_COL[s], alpha=.85)
    ax.text(-0.02, y0 + len(v) / 2, f"{s} (n={len(v)})", ha="right", va="center", fontsize=8.6); y0 += len(v) + 15
ax.axvline(sv.mean(), color="#D55E00", ls="--"); ax.text(sv.mean() + .01, y0 - 5, f"mean = {sv.mean():.3f}", color="#D55E00", fontsize=9)
ax.set_yticks([]); ax.set_xlabel("Silhouette coefficient"); ax.set_xlim(-0.6, 0.9)
ax.set_title(f"Silhouette plot for the K-Means (k = {K}) solution", loc="left"); ax.grid(axis="y", visible=False)
save(fig, "fig10_silhouette.png")

# =============================================================== 12. PROFILES
P_ = df.groupby("segment").agg(
    Products=("price", "size"), Median_price=("price", "median"), Median_sold=("sold", "median"),
    Mean_sold=("sold", "mean"), Last_one_or_low_pct=("stock_band", lambda s: (s <= 1).mean() * 100),
    High_stock_pct=("stock_band", lambda s: (s == 3).mean() * 100),
    EDP_parfum_pct=("concentration", lambda s: s.isin(["Eau de Parfum", "Parfum / Extrait"]).mean() * 100),
    International_pct=("international", lambda s: s.mean() * 100), Revenue=("revenue", "sum")).reindex(SEG_ORDER)
P_["Product_share_pct"] = P_["Products"] / P_["Products"].sum() * 100
P_["Revenue_share_pct"] = P_["Revenue"] / P_["Revenue"].sum() * 100
P_.round(2).to_csv(OUT + "tab_profiles.csv")
top_brands = df.groupby("segment")["brand"].agg(lambda s: ", ".join(s.value_counts().head(3).index)).reindex(SEG_ORDER)
top_brands.to_csv(OUT + "tab_brands.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.38})
zc = pd.DataFrame(km.cluster_centers_, columns=["log price", "log sold", "stock band"]).rename(index=NAMES).reindex(SEG_ORDER)
sns.heatmap(zc, annot=True, fmt="+.2f", cmap="RdBu_r", center=0, vmin=-2, vmax=2, linewidths=.8,
            cbar_kws={"label": "Centroid (z-score)"}, ax=a1)
a1.set_title("(a) Segment centroids on the scaled features", loc="left", fontsize=11); a1.set_ylabel(""); a1.grid(False)
x = np.arange(len(SEG_ORDER)); w = .38
a2.bar(x - w / 2, P_["Product_share_pct"], w, color="#c5ced8", label="% of products")
a2.bar(x + w / 2, P_["Revenue_share_pct"], w, color=[SEG_COL[s] for s in SEG_ORDER], label="% of revenue (price x sold)")
for i, (p, r) in enumerate(zip(P_["Product_share_pct"], P_["Revenue_share_pct"])):
    a2.text(i - w / 2, p + 1.5, f"{p:.0f}%", ha="center", fontsize=8.5); a2.text(i + w / 2, r + 1.5, f"{r:.0f}%", ha="center", fontsize=8.5, fontweight="bold")
a2.set_xticks(x); a2.set_xticklabels(["Stock-out\nrisk BS", "Well-\nstocked BS", "Premium\nniche", "Over-\nstocked", "Low-stock\nslow"], fontsize=8.3)
a2.set_ylim(0, 68); a2.legend(frameon=False, fontsize=8.5, loc="upper right"); a2.grid(axis="x", visible=False)
a2.set_title("(b) Share of catalogue vs share of revenue", loc="left", fontsize=11)
save(fig, "fig11_profiles.png")

# price-sold scatter in original units (business view)
fig, ax = plt.subplots(figsize=(9.5, 5.2))
for s in SEG_ORDER:
    m = df["segment"] == s
    ax.scatter(df.loc[m, "price"], df.loc[m, "sold"] + 1, s=10 + df.loc[m, "stock_band"] * 10, alpha=.6, color=SEG_COL[s], label=s)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Price (USD, log scale)"); ax.set_ylabel("Units sold + 1 (log scale)")
ax.set_title("Products by price and units sold, coloured by K-Means segment (marker size = stock band)", loc="left", fontsize=11.5)
ax.legend(frameon=False, fontsize=9, markerscale=1.5)
save(fig, "fig12_business.png")

# =============================================================== 13. SUMMARY FOR THE REPORT
summary = dict(n_raw=len(raw), n_clean=len(df), dropped=dropped, price_rec=n_price_rec, sold_rec=n_sold_rec,
               cad=int(cad.sum()), raw_cats=int(raw["Catatory"].nunique()), evr=list(np.round(evr, 3)),
               minpts=MINPTS, eps=EPS, names={int(k): v for k, v in NAMES.items()}, km_k=kdf.to_dict("records"),
               skew={c: [round(df[c].skew(), 2), round(np.log1p(df[c]).skew(), 2)] for c in ["price", "sold"]},
               total_revenue=round(df["revenue"].sum()), sold_top10pct_share=round(df["sold"].nlargest(int(.1 * len(df))).sum() / df["sold"].sum() * 100, 1))
json.dump(summary, open(OUT + "summary.json", "w"), indent=1, default=str)
for f in ["tab_transform", "tab_kmeans_k", "tab_linkage", "tab_compare", "tab_ari", "tab_gmm_bic", "tab_loadings", "tab_profiles", "tab_brands"]:
    print(f"\n--- {f}\n" + pd.read_csv(OUT + f + ".csv").to_string())
print(json.dumps(summary, indent=1, default=str))
print(gdf.to_string())



--- tab_transform
                                Feature set                       Scaling  Dims  Best k  Silhouette  Davies-Bouldin  Smallest cluster  ARI vs stock band  ARI vs concentration
0              A: core (price, sold, stock)      StandardScaler (z-score)     3       7       0.311           1.035                90               0.27                  0.03
1              A: core (price, sold, stock)     RobustScaler (median/IQR)     3       2       0.293           1.336               460               0.09                  0.02
2              A: core (price, sold, stock)  MinMaxScaler (normalise 0-1)     3       2       0.450           0.890               377               0.61                  0.01
3  B: core + concentration one-hot + origin      StandardScaler (z-score)     9       6       0.430           0.835                27               0.01                  0.85
4                   A without log transform      StandardScaler (z-score)     3       8       0.476       